# Rajaraman et al. (2018) patient-level baseline reproduction

## Objective
Reproduce the supplied base paper's ResNet-50 feature-extractor baseline on the exact NIH/NLM malaria cell-image release using a verified patient-grouped five-fold protocol. This notebook uses only code in `src/`; it fetches or verifies the official release, runs the audit before model work, and writes all tables/figures to `results/baseline/`. Parasitized is the positive class.

## Base paper
The M2 reference is Rajaraman et al. (2018), *Pre-trained convolutional neural networks as feature extractors toward improved malaria parasite detection in thin blood smear images* (PeerJ 6:e4568). The supplied project summary reports patient-level five-fold CV and Table 6 values of accuracy 0.959, sensitivity 0.947, specificity 0.972, and AUC 0.991 for the proposed model. See `milestones/M2/reproduction_notes.md` for reported versus approximated details.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(parent for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (parent / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Project root: {ROOT}')

Project root: /Users/amankumar/Desktop/CV


## Dataset
The notebook obtains only NLM's official `cell_images.zip` and its two official Patient-ID-to-cell mapping CSVs if they are absent. It does not use Kaggle, TensorFlow Dataset copies, or another malaria dataset. Archive source/version evidence and SHA-256 hashes are saved in `results/dataset_source.json`.

In [ ]:
from src.dataset import audit_dataset

audit = audit_dataset(ROOT, fetch_missing=True)
assert audit['passed'], 'Audit failed; model work is intentionally blocked.'
audit

{'passed': True,
 'expected_total': 27558,
 'observed_total': 27558,
 'expected_per_class': 13779,
 'class_counts': {'Uninfected': 13779, 'Parasitized': 13779},
 'official_mapping_entries': {'Parasitized': 151, 'Uninfected': 201},
 'unique_official_patient_ids_across_classes': 201,
 'cells_with_no_official_patient_id': 0,
 'corrupt_images': 0,
 'mapping_references_absent_images': 0,
 'images_absent_from_mapping': 0,
 'duplicate_mapping_assignments': 0,
 'byte_identical_duplicate_sets': 0,
 'cross_label_duplicate_sets': 0,
 'image_dimensions': {'100x100': 26,
  '100x103': 37,
  '100x106': 25,
  '100x109': 22,
  '100x112': 24,
  '100x115': 19,
  '100x118': 34,
  '100x121': 15,
  '100x124': 16,
  '100x127': 11,
  '100x130': 5,
  '100x133': 6,
  '100x136': 10,
  '100x139': 4,
  '100x142': 4,
  '100x145': 3,
  '100x148': 6,
  '100x151': 2,
  '100x154': 1,
  '100x157': 3,
  '100x163': 2,
  '100x166': 1,
  '100x172': 1,
  '100x175': 2,
  '100x76': 1,
  '100x79': 1,
  '100x82': 2,
  '100x85': 

## Dataset verification
The audit checks directory structure, the 13,779/13,779 class counts, every PNG's decodability, official mapping completeness, byte-identical duplicates, and cross-label duplicates. It is the gate for all training.

In [ ]:
display(pd.read_csv(ROOT / 'results/baseline/dataset_class_counts.csv'))
display(pd.read_csv(ROOT / 'results/baseline/dataset_mapping_issues.csv'))
display(pd.read_csv(ROOT / 'results/baseline/duplicate_images.csv'))

,class_name,label,image_count
0,Uninfected,0,13779
1,Parasitized,1,13779


,issue,class_name,filename,patient_id_first,patient_id_second


,sha256,copies,classes,paths


## Patient/cell mapping
The split group is the exact official NLM mapping key, not a random or filename-derived ID. The mapping summary is saved for auditing and M4 reuse. It does not reproduce paper1's unpublished fold membership.

In [ ]:
patient_summary = pd.read_csv(ROOT / 'results/baseline/patient_cell_mapping_summary.csv')
print('Official mapping IDs:', patient_summary.patient_id.nunique())
display(patient_summary.head())

Official mapping IDs: 201


,patient_id,class_name,cell_count
0,C100P61ThinF,Parasitized,57
1,C100P61ThinF,Uninfected,68
2,C101P62ThinF,Parasitized,314
3,C101P62ThinF,Uninfected,71
4,C102P63ThinF,Parasitized,35


## Preprocessing
Reported source detail: model-compatible resize and mean normalization. M2 uses 224×224 bilinear resizing and ImageNet normalization because the frozen ImageNet-pretrained ResNet-50 requires a documented convention; this is explicitly logged as an approximation. No augmentation is used because the supplied paper1 source does not specify an augmentation recipe.

## Cross-validation / splitting
The paper reports patient-level five-fold CV. M2 makes a deterministic patient-grouped, class-stratified five-fold assignment from the official mapping keys (seed 42), saves it, and asserts zero group overlap in every fold.

In [ ]:
from src.dataset import load_manifest
from src.splits import make_grouped_folds

manifest = load_manifest(ROOT)
assignments = make_grouped_folds(manifest, ROOT, n_splits=5, seed=42)
fold_summary = pd.read_csv(ROOT / 'results/baseline/fold_summary.csv')
assert (fold_summary.patient_overlap == 0).all()
display(fold_summary)

,fold,train_samples,test_samples,train_parasitized,train_uninfected,test_parasitized,test_uninfected,train_patients,test_patients,patient_overlap
0,1,22061,5497,11027,11034,2752,2745,161,40,0
1,2,22038,5520,11007,11031,2772,2748,161,40,0
2,3,22014,5544,11045,10969,2734,2810,160,41,0
3,4,22075,5483,11034,11041,2745,2738,161,40,0
4,5,22044,5514,11003,11041,2776,2738,161,40,0


## Model configuration
The reported base approach is a pretrained ResNet-50 feature extractor with a fully connected classifier and dropout 0.5. M2 freezes torchvision's ImageNet ResNet-50, extracts final global-average-pooled 2,048-dimensional features, then trains a fresh 2048→512→1 classifier with ReLU/dropout 0.5 in each fold. The exact paper layer, hidden width, optimization search ranges, seed, batch size, and original fold assignments are unavailable in the supplied source; all substitutions are recorded in `REPRODUCTION_NOTES.md`.

## Training / feature extraction
Each test fold remains unseen: its classifier is trained only on the other four folds with fixed SGD settings (no test-fold tuning). Feature extraction is cached after a complete ordered pass to make re-execution practical.

In [ ]:
from src.run_baseline import run_baseline

run = run_baseline(ROOT)
display(run['baseline'])

,run_id,protocol,positive_class,metric,mean,standard_deviation,n_folds,measured_at_utc
0,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,accuracy,0.938978,0.019460,5,2026-10-03T17:22:55.052013+00:00
1,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,sensitivity_recall,0.942399,0.033580,5,2026-10-03T17:22:55.054015+00:00
2,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,specificity,0.935640,0.013029,5,2026-10-03T17:22:55.054066+00:00
3,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,auc,0.983221,0.008648,5,2026-10-03T17:22:55.054106+00:00
4,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,f1,0.939016,0.020341,5,2026-10-03T17:22:55.054151+00:00


## Prediction
Each image receives exactly one out-of-fold probability for Parasitized. The complete trace is `results/per_sample_predictions.csv`, with the fold assignment and official mapping ID.

## Evaluation metrics
Measured metrics are accuracy, sensitivity/recall (Parasitized positive), specificity, ROC AUC, and F1. The threshold for binary metrics is 0.5; AUC uses probabilities.

## Per-fold results


In [ ]:
display(pd.read_csv(ROOT / 'results/baseline/fold_results.csv'))

,accuracy,sensitivity_recall,specificity,auc,f1,true_negative,false_positive,false_negative,true_positive,fold,train_final_bce,test_samples
0,0.936875,0.953125,0.920583,0.983518,0.937958,2527,218,129,2623,1,0.142801,5497
1,0.906341,0.882756,0.930131,0.968392,0.904454,2556,192,325,2447,2,0.134554,5520
2,0.949315,0.963789,0.935231,0.987529,0.949378,2628,182,99,2635,3,0.148508,5544
3,0.955681,0.955191,0.956172,0.990370,0.955714,2618,120,123,2622,4,0.154760,5483
4,0.946681,0.957133,0.936085,0.986297,0.947575,2563,175,119,2657,5,0.150764,5514


## Mean ± SD
The primary M2 summary is the unweighted mean and sample standard deviation across the five patient-grouped test folds. The saved `baseline_results.csv` contains measured values only.

In [ ]:
display(pd.read_csv(ROOT / 'results/baseline/baseline_results.csv'))

,run_id,protocol,positive_class,metric,mean,standard_deviation,n_folds,measured_at_utc
0,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,accuracy,0.938978,0.019460,5,2026-10-03T17:22:55.052013+00:00
1,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,sensitivity_recall,0.942399,0.033580,5,2026-10-03T17:22:55.054015+00:00
2,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,specificity,0.935640,0.013029,5,2026-10-03T17:22:55.054066+00:00
3,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,auc,0.983221,0.008648,5,2026-10-03T17:22:55.054106+00:00
4,m2_rajaraman_resnet50_feature_extractor,patient-grouped stratified 5-fold CV (official...,Parasitized,f1,0.939016,0.020341,5,2026-10-03T17:22:55.054151+00:00


## Comparison with paper
Published values are deliberately stored separately in `published_baseline.csv`; `published_vs_measured.csv` is the comparison. Numerical agreement is not expected to be exact because the official mapping supports patient grouping but does not disclose the paper's fold membership, and several model/training details are absent from the supplied project source.

In [ ]:
display(pd.read_csv(ROOT / 'results/baseline/published_vs_measured.csv'))
display(pd.read_csv(ROOT / 'results/baseline/per_sample_predictions.csv').head())

,source,paper_location,protocol,metric,published_value,measured_mean,measured_fold_sd,measured_minus_published
0,"Rajaraman et al. 2018, supplied paper1 summary","Table 6, proposed model, patient level",patient-level 5-fold CV,accuracy,0.959,0.938978,0.019460,-0.020022
1,"Rajaraman et al. 2018, supplied paper1 summary","Table 6, proposed model, patient level",patient-level 5-fold CV,sensitivity_recall,0.947,0.942399,0.033580,-0.004601
2,"Rajaraman et al. 2018, supplied paper1 summary","Table 6, proposed model, patient level",patient-level 5-fold CV,specificity,0.972,0.935640,0.013029,-0.036360
3,"Rajaraman et al. 2018, supplied paper1 summary","Table 6, proposed model, patient level",patient-level 5-fold CV,auc,0.991,0.983221,0.008648,-0.007779


,relative_path,filename,class_name,label,patient_id,fold,probability_parasitized,prediction
0,cell_images/Parasitized/C100P61ThinF_IMG_20150...,C100P61ThinF_IMG_20150918_144104_cell_162.png,Parasitized,1,C100P61ThinF,2,0.998670,1
1,cell_images/Parasitized/C100P61ThinF_IMG_20150...,C100P61ThinF_IMG_20150918_144104_cell_163.png,Parasitized,1,C100P61ThinF,2,0.946270,1
2,cell_images/Parasitized/C100P61ThinF_IMG_20150...,C100P61ThinF_IMG_20150918_144104_cell_164.png,Parasitized,1,C100P61ThinF,2,0.990998,1
3,cell_images/Parasitized/C100P61ThinF_IMG_20150...,C100P61ThinF_IMG_20150918_144104_cell_165.png,Parasitized,1,C100P61ThinF,2,0.998588,1
4,cell_images/Parasitized/C100P61ThinF_IMG_20150...,C100P61ThinF_IMG_20150918_144104_cell_166.png,Parasitized,1,C100P61ThinF,2,0.697198,1


## Reproduction discussion
This is a full M2 run on the exact official release, with zero official-Patient-ID overlap between train and test. The comparison should be read as an approximate reproduction, not a claim of paper-exact equivalence: our grouped fold assignment, feature layer, ImageNet weight release/normalization, classifier width, and fixed optimizer settings differ where paper1's supplied project summary does not give reproducible detail.

## Limitations
This is a public segmented-cell research experiment, not a diagnostic or clinical-performance claim. It is limited to this single P. falciparum thin-smear cell dataset, public labels and official mapping keys, no external dataset, and no recovery of paper1's original fold identities. M4 is intentionally not executed in this notebook.